### Testando instalação de libs

In [1]:
!pip install minio

### Import das libs

In [13]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from minio import Minio
from minio.error import S3Error

### Testando PySpark e driver do Postgresql

In [14]:
spark = SparkSession.builder \
    .appName("CSV to PostgreSQL") \
    .config("spark.driver.extraClassPath", "/home/jovyan/notebooks/jars/postgresql-42.6.0.jar") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .getOrCreate()

#csv_file_path = "/home/jovyan/notebooks/example.csv"

csv_file_path = "/home/jovyan/notebooks/_temporary_blobs"
df = spark.read.parquet("/home/jovyan/notebooks/_temporary_blobs", header=True)

#csv_schema = StructType([
#    StructField("id", IntegerType(), True),
#    StructField("name", StringType(), True),    
#])

#df = spark.read.csv(csv_file_path, header=True
 #                   , schema=csv_schema
#                   )

postgres_url = "jdbc:postgresql://postgres:5432/prova"
postgres_properties = {
    "user": "postgres",
    "password": "password",
    "driver": "org.postgresql.Driver"
}
postgres_table = "dim_alunos"

df.write \
    .format("jdbc") \
    .option("url", postgres_url) \
    .option("dbtable", postgres_table) \
    .option("user", postgres_properties["user"]) \
    .option("password", postgres_properties["password"]) \
    .option("driver", postgres_properties["driver"]) \
    .mode("overwrite") \
    .save()

print('Dados inseridos no banco de dados!')

df.show()

spark.stop()

Dados inseridos no banco de dados!
+---+--------------------+---------------+--------------------+
| id|                nome|data_nascimento|            etl_date|
+---+--------------------+---------------+--------------------+
|  1|      Agatha Barbosa|     1961-01-07|2024-01-13 19:37:...|
|  2|        Nathan Pinto|     1998-01-08|2024-01-13 19:37:...|
|  3|        Renan Aragão|     1979-01-05|2024-01-13 19:37:...|
|  4|       Kevin Rezende|     1978-10-21|2024-01-13 19:37:...|
|  5|        Bruna Barros|     1966-11-05|2024-01-13 19:37:...|
|  6|   Sra. Evelyn Silva|     1969-04-21|2024-01-13 19:37:...|
|  7|     Isabella Farias|     1993-04-06|2024-01-13 19:37:...|
|  8|      Daniel Rezende|     2002-07-01|2024-01-13 19:37:...|
|  9|  João Lucas da Mata|     1964-12-30|2024-01-13 19:37:...|
| 10|         Joana Nunes|     1977-12-22|2024-01-13 19:37:...|
| 11|         Renan Alves|     1963-05-31|2024-01-13 19:37:...|
| 12|        Alícia Sales|     1981-09-09|2024-01-13 19:37:...|
| 13|

### Testando a conexão com Minio

In [4]:
minio_endpoint = 'minio:9000'
minio_access_key = 'minioaccesskey'
minio_secret_key = 'miniosecretkey'
bucket_name = 'raw'
local_file_path = '/home/jovyan/notebooks/example.csv'
minio_object_name = 'example.csv'

minio_client = Minio(minio_endpoint, access_key=minio_access_key, secret_key=minio_secret_key, secure=False)

if not minio_client.bucket_exists(bucket_name):
    minio_client.make_bucket(bucket_name)

try:
    minio_client.fput_object(bucket_name, minio_object_name, local_file_path)
    print(f"Arquivo '{minio_object_name}' enviado com sucesso para o MinIO.")
except S3Error as e:
    print(f"Erro ao enviar o arquivo para o MinIO: {e}")

Arquivo 'example.csv' enviado com sucesso para o MinIO.
